<a href="https://colab.research.google.com/github/dzikrilazizhakim/Semester-5/blob/main/Computer%20Vision%20Petrain%20Model%20Dzikril_Aziz_Hakim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

KONFIGURASI DAN SET UP

In [ ]:
import copy
import random
import time

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import ConcatDataset, DataLoader, Dataset
from torchvision import datasets, models, transforms
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import torchvision

SEED = 42

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

CONFIG = {
    "img_size": 128,        # citra CIFAR 32x32 di-resize; 224 lebih akurat tetapi lebih lambat
    "batch_size": 64,
    "epochs": 10,           # instruksi: minimum 5-10 epoch
    "lr_backbone": 1e-4,    # fine-tuning: learning rate rendah untuk lapisan pretrained
    "lr_head": 1e-3,        # head baru (bobot acak) boleh sedikit lebih besar
    "weight_decay": 1e-4,
    "dropout": 0.2,
    "train_ratio": 0.8,     # 80% training, 20% validation
    "subset_fraction": 1.0, # isi < 1.0 (mis. 0.1) hanya untuk uji cepat
    "num_workers": 2,
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"   # mixed precision hanya untuk GPU
print("Device:", device, "| torch", torch.__version__, "| torchvision", torchvision.__version__)
print("Konfigurasi:", CONFIG)

PERSIAPAN DATASET (TRAINING 80% / VALIDATION 20%)

In [ ]:
DATA_DIR = "./data"
raw_train = datasets.CIFAR10(DATA_DIR, train=True, download=True)   # citra PIL, transform ditambahkan kemudian
raw_test = datasets.CIFAR10(DATA_DIR, train=False, download=True)
CLASSES = raw_train.classes
NUM_CLASSES = len(CLASSES)

full_ds = ConcatDataset([raw_train, raw_test])
all_targets = np.array(list(raw_train.targets) + list(raw_test.targets))

indices = np.arange(len(full_ds))
if CONFIG["subset_fraction"] < 1.0:
    indices, _ = train_test_split(indices, train_size=CONFIG["subset_fraction"],
                                  stratify=all_targets, random_state=SEED)

# Split 80/20, stratified agar distribusi kelas di kedua himpunan seimbang
train_idx, val_idx = train_test_split(indices, train_size=CONFIG["train_ratio"],
                                      stratify=all_targets[indices], random_state=SEED)

n_total = len(indices)
print(f"Total citra : {n_total}")
print(f"Training    : {len(train_idx)} ({len(train_idx) / n_total:.0%})")
print(f"Validation  : {len(val_idx)} ({len(val_idx) / n_total:.0%})")
print("Kelas:", CLASSES)

In [ ]:
IMG = CONFIG["img_size"]
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]  # statistik ImageNet, sesuai bobot pretrained

class TransformSubset(Dataset):
    """Subset dengan transform sendiri, sehingga train dan validation bisa memakai augmentasi berbeda."""
    def __init__(self, base, idx, transform):
        self.base, self.idx, self.transform = base, idx, transform

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, i):
        img, label = self.base[int(self.idx[i])]
        return self.transform(img), label

train_tf = transforms.Compose([
    transforms.Resize((IMG, IMG)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG, IMG)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

train_ds = TransformSubset(full_ds, train_idx, train_tf)
val_ds = TransformSubset(full_ds, val_idx, eval_tf)

pin = device.type == "cuda"
train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True,
                          num_workers=CONFIG["num_workers"], pin_memory=pin)
val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"] * 2, shuffle=False,
                        num_workers=CONFIG["num_workers"], pin_memory=pin)

counts = np.bincount(all_targets[train_idx], minlength=NUM_CLASSES)
print("Distribusi kelas (train):", dict(zip(CLASSES, counts.tolist())))

In [ ]:
def denormalize(x):
    mean = torch.tensor(MEAN).view(3, 1, 1)
    std = torch.tensor(STD).view(3, 1, 1)
    return (x.cpu() * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

# Pemeriksaan visual: pastikan label dan augmentasi sudah benar
imgs, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 6, figsize=(12, 4.8))
for ax, img, lab in zip(axes.ravel(), imgs, labels):
    ax.imshow(denormalize(img))
    ax.set_title(CLASSES[lab], fontsize=10)
    ax.axis("off")
plt.suptitle("Contoh citra training (setelah augmentasi)")
plt.tight_layout()
plt.show()

LOAD PRETRAINED MODEL & MODIFIKASI TOP LAYER

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

print("Classifier asli :", model.fc)
in_features = model.fc.in_features   # 512 untuk ResNet-18
model.fc = nn.Sequential(
    nn.Dropout(CONFIG["dropout"]),
    nn.Linear(in_features, NUM_CLASSES),
)
print("Classifier baru :", model.fc)
model = model.to(device)

FINE-TUNING DENGAN LEARNING RATE RENDAH

In [ ]:
head_params = list(model.fc.parameters())
head_ids = {id(p) for p in head_params}
backbone_params = [p for p in model.parameters() if id(p) not in head_ids]

optimizer = torch.optim.Adam(
    [
        {"params": backbone_params, "lr": CONFIG["lr_backbone"]},
        {"params": head_params, "lr": CONFIG["lr_head"]},
    ],
    weight_decay=CONFIG["weight_decay"],
)
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG["epochs"])

n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_all = sum(p.numel() for p in model.parameters())
print(f"LR backbone = {CONFIG['lr_backbone']} | LR head = {CONFIG['lr_head']}")
print(f"Parameter yang dilatih: {n_trainable:,} / {n_all:,}")

In [ ]:
scaler = torch.amp.GradScaler(device.type, enabled=use_amp)

def run_epoch(loader, train):
    model.train(train)
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.autocast(device_type=device.type, enabled=use_amp):
                logits = model(x)
                loss = criterion(logits, y)
            if train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            total_loss += loss.item() * x.size(0)
            correct += (logits.argmax(1) == y).sum().item()
            n += x.size(0)
    return total_loss / n, correct / n

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_acc, best_state = 0.0, None
start = time.time()

for epoch in range(1, CONFIG["epochs"] + 1):
    t0 = time.time()
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    va_loss, va_acc = run_epoch(val_loader, train=False)
    scheduler.step()

    history["train_loss"].append(tr_loss)
    history["train_acc"].append(tr_acc)
    history["val_loss"].append(va_loss)
    history["val_acc"].append(va_acc)

    if va_acc > best_acc:
        best_acc = va_acc
        best_state = copy.deepcopy(model.state_dict())

    print(f"Epoch {epoch:02d}/{CONFIG['epochs']} | "
          f"train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
          f"val loss {va_loss:.4f} acc {va_acc:.4f} | {time.time() - t0:.0f}s")

print(f"Selesai dalam {(time.time() - start) / 60:.1f} menit. Best validation accuracy: {best_acc:.4f}")

torch.save(best_state, "best_resnet18_cifar10.pt")
model.load_state_dict(best_state)   # gunakan bobot terbaik untuk evaluasi

EVALUASI: CLASSIFICATION REPORT & CONFUSION MATRIX

In [ ]:
C_TRAIN, C_VAL = "#2a78d6", "#eb6834"   # dua slot kategorikal dengan jarak warna aman untuk CVD
epochs_range = range(1, len(history["train_loss"]) + 1)
best_epoch = int(np.argmax(history["val_acc"])) + 1

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.6))

ax1.plot(epochs_range, history["train_loss"], color=C_TRAIN, lw=2, marker="o", ms=6, label="Train")
ax1.plot(epochs_range, history["val_loss"], color=C_VAL, lw=2, marker="o", ms=6, label="Validation")
ax1.set_title("Loss per epoch")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Cross-entropy loss")
ax1.grid(alpha=0.3)
ax1.legend(frameon=False)

ax2.plot(epochs_range, history["train_acc"], color=C_TRAIN, lw=2, marker="o", ms=6, label="Train")
ax2.plot(epochs_range, history["val_acc"], color=C_VAL, lw=2, marker="o", ms=6, label="Validation")
ax2.axvline(best_epoch, color="#52514e", ls="--", lw=1, label=f"Best epoch ({best_epoch})")
ax2.set_title("Accuracy per epoch")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.grid(alpha=0.3)
ax2.legend(frameon=False)

plt.tight_layout()
plt.show()

In [ ]:
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(8, 7))
# Sequential single-hue colormap: nilai lebih besar = warna lebih gelap
ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(
    ax=ax, cmap="Blues", xticks_rotation=45, colorbar=False)
ax.set_title("Confusion matrix (validation set)")
plt.tight_layout()
plt.show()

In [ ]:
imgs, labels = next(iter(val_loader))
with torch.no_grad(), torch.autocast(device_type=device.type, enabled=use_amp):
    preds = model(imgs.to(device)).argmax(1).cpu()

fig, axes = plt.subplots(2, 6, figsize=(12, 4.8))
for ax, img, p, t in zip(axes.ravel(), imgs, preds, labels):
    ax.imshow(denormalize(img))
    mark = "✓" if int(p) == int(t) else "✗"   # ikon + teks, tidak hanya warna
    ax.set_title(f"{mark} pred: {CLASSES[p]}\ntrue: {CLASSES[t]}", fontsize=9)
    ax.axis("off")
plt.suptitle("Contoh prediksi pada data validation")
plt.tight_layout()
plt.show()